# Centaur-70B scale check — surprise-rep, no RT

**Question:** does the *full* Centaur (70B) carry more transferable cross-task individual
signal than Minitaur (8B)? If not, the thin residual is a property of the **choice-only**
objective, not model size — which is itself a result.

**Method:** training-free **surprise reps** (per-response NLL + entropy under the raw model)
→ cross-task identification matrix. No fine-tuning, no RT. Runs raw **8B** and raw **70B**
with the *same* pipeline and compares within/across-domain identification.

Needs a big GPU (H100 / A100-80G). 70B loads in 4-bit (~35–40 GB). Profiles are cached to
Drive per task, so a disconnect resumes where it stopped.

In [ ]:
# 1) setup (fresh runtime)
from google.colab import drive
drive.mount('/content/drive')

import os
if os.path.isdir('/content/sro-minitaur-transfer'):
    !cd /content/sro-minitaur-transfer && git pull
else:
    !git clone https://github.com/YifeiCAO/sro-minitaur-transfer.git /content/sro-minitaur-transfer

!pip -q install peft bitsandbytes scikit-learn

In [ ]:
# 2) discover Marcel Binz's actual repos -> read off the exact Centaur-70B id
from huggingface_hub import list_models
for mi in list_models(author='marcelbinz'):
    print(mi.id)

# 3) HF login: REQUIRED only if the 70B is a LoRA adapter on gated meta-llama/Llama-3.1-70B
#    (also silences the rate-limit warning). Uncomment and paste a token if needed.
# from huggingface_hub import login; login(token='hf_xxx')

## Set the 70B repo

From the printed list, set `CENTAUR_70B`:
- **Merged & public** (like Minitaur-8B) → `IS_ADAPTER = False` (simplest).
- **LoRA adapter on gated Llama-70B** → `IS_ADAPTER = True` (accept the Llama-3.1-70B
  license on HF and `login()` above; this also downloads the ~140 GB base).

In [ ]:
CENTAUR_70B = 'marcelbinz/Llama-3.1-Centaur-70B'   # <-- set from the list above
IS_ADAPTER  = False                                # True if it's an adapter on gated Llama-70B
BASE_LLM    = 'meta-llama/Llama-3.1-70B'           # only used when IS_ADAPTER

ADAPTER_FLAGS = f'--base-is-adapter --base-llm {BASE_LLM}' if IS_ADAPTER else ''
print('70B:', CENTAUR_70B, '| adapter:', IS_ADAPTER)

In [ ]:
# 4) matched 8B baseline: RAW Minitaur (no SRO adapter). Fast.
#    Caches profiles to results/surprise_8b_raw/ on Drive (resumable).
!cd /content/sro-minitaur-transfer && python scripts/build_surprise_matrix.py \
    --raw --base-model marcelbinz/Llama-3.1-Minitaur-8B \
    --subset starting_subset --max-seq-len 4096 --batch-tokens 32768 --tag 8b_raw

In [ ]:
# 5) the scale check: RAW Centaur-70B. Main run.
#    --batch-tokens 32768 packs many sessions per forward to fill the idle ~50 GB
#    (70B 4-bit is ~40 GB; this uses the rest -> big speedup). Lower it if you OOM.
#    Caches profiles to results/surprise_70b_raw/ on Drive (resume by re-running).
!cd /content/sro-minitaur-transfer && python scripts/build_surprise_matrix.py \
    --raw --base-model {CENTAUR_70B} {ADAPTER_FLAGS} \
    --subset starting_subset --max-seq-len 4096 --batch-tokens 32768 --tag 70b_raw

In [ ]:
# 6) compare
import json
r = '/content/drive/MyDrive/sro_minitaur/results'
for tag in ['8b_raw', '70b_raw']:
    try:
        s = json.load(open(f'{r}/surprise_matrix_summary_{tag}.json'))
        print(f"{tag:8s}  within={s['within']:.3f}  across={s['across']:.3f}  chance={s['chance']:.3f}")
    except FileNotFoundError:
        print(f'{tag}: not found yet')
print('\nwithin(70B) >> within(8B)  -> scale helps, worth fine-tuning 70B.')
print('within(70B) ~= within(8B)  -> choice-only ceiling: thinness is the objective, not size.')